# Benchmarking VQE (k-UpCCSD) — CH₄ STO-3G
**Circuit-based Qiskit + FakeSherbrooke Noise Model**

| Parameter | Nilai |
|---|---|
| Molekul | CH₄ (C-H bond, Td symmetry) |
| Basis set | STO-3G (9 orbital → 18 qubit) |
| Ansatz | k-UpCCSD (k lapisan, Paired Double; Trotterisasi orde-1, Qiskit QuantumCircuit) |
| Optimizer | L-BFGS-B (gradient via finite differences 2-point) |
| Noise model | FakeSherbrooke (IBM 127-qubit) |
| Shots | 16 384 |
| Referensi | FCI (diagonalisasi matriks eksak) |


In [2]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 – IMPORTS
# ════════════════════════════════════════════════════════════════════
import os, re, glob, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from IPython.display import display
from scipy.sparse.linalg import eigsh

warnings.filterwarnings('ignore')

# ── Quantum chemistry ─────────────────────────────────────────────
from pyscf import gto, scf, ao2mo
from openfermion import (FermionOperator, normal_ordered, jordan_wigner,
                         get_sparse_operator, count_qubits)

# ── Qiskit core ───────────────────────────────────────────────────
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

# ── Qiskit Aer ────────────────────────────────────────────────────
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import Estimator as AerEstimator

# ── IBM Fake Backend (FakeMumbai 27-qubit) ────────────────────────
try:
    from qiskit_ibm_runtime.fake_provider import FakeMumbaiV2 as _FakeBackendCls
    _FAKE_SRC = 'qiskit_ibm_runtime.fake_provider'
except ImportError:
    try:
        from qiskit.providers.fake_provider import FakeMumbaiV2 as _FakeBackendCls
        _FAKE_SRC = 'qiskit.providers.fake_provider (V2)'
    except ImportError:
        from qiskit.providers.fake_provider import FakeMumbai as _FakeBackendCls
        _FAKE_SRC = 'qiskit.providers.fake_provider (V1)'

print(f'✅ FakeMumbai dimuat dari : {_FAKE_SRC}')
print('✅ Semua library berhasil diimport')

✅ FakeMumbai dimuat dari : qiskit_ibm_runtime.fake_provider
✅ Semua library berhasil diimport


In [3]:
# ════════════════════════════════════════════════════════════════════
# CELL 2 – KONFIGURASI
# ════════════════════════════════════════════════════════════════════
ORCA_OUTPUT_DIR  = './input_2'

# Ikatan C-H tetrahedral; ekuilibrium eksperimen ≈ 1.09 Å
# Dense di sekitar ekuilibrium, sparse di dissosiasi
BOND_LENGTHS_VQE = np.round(np.concatenate([
    np.arange(0.80, 1.20, 0.05),   # 0.80–1.15 (dense, 8 titik)
    np.arange(1.20, 2.01, 0.10),   # 1.20–2.00 (sparse, 9 titik)
]), 2)

# ── Pilihan optimizer dan ansatz ─────────────────────────────────────
METHOD_VQE   = 'L-BFGS-B'   # gradient-based; gunakan 'COBYLA' untuk gradient-free
K_LAYERS     = 1            # jumlah lapisan k-UpCCSD  (k=1,2,3, …)
N_RESTARTS   = 1            # CH4 sangat mahal; 1 restart sudah cukup di HPC
SEED         = 42
N_ELECTRONS  = 10           # CH₄ : 6(C) + 4×1(H)
N_QUBITS     = 18           # STO-3G: 9 orbital spasial × 2 spin
N_SHOTS      = 2048         # Shot untuk noise simulator
BACKEND_NAME = 'FakeSherbrooke (127-qubit, IBM)'

# ── Ringkasan reduksi parameter vs UCCSD ─────────────────────────────
# k-UpCCSD  (k=1): singles=80, paired_doubles=5×4=20  → 100 param/layer
# UCCSD full     : singles=80, all_doubles=C(10,2)×C(8,2)=1260 → 1340 param
# Reduksi        : ≈ 13× lebih sedikit parameter

print('📌 Konfigurasi Benchmark CH₄ — k-UpCCSD + L-BFGS-B + Noise:')
print(f'   Molekul      : CH₄  (C-H bond, simetri Td)')
print(f'   Basis set    : STO-3G  →  {N_QUBITS} qubit,  {N_ELECTRONS} elektron')
print(f'   Bond lengths : {BOND_LENGTHS_VQE[0]:.2f}–{BOND_LENGTHS_VQE[-1]:.2f} Å'
      f'  ({len(BOND_LENGTHS_VQE)} titik)')
print(f'   Optimizer    : {METHOD_VQE}  |  Restarts : {N_RESTARTS}  |  Seed : {SEED}')
print(f'   Backend noise: {BACKEND_NAME}  |  Shots : {N_SHOTS:,}')
print(f'   Ansatz       : {K_LAYERS}-UpCCSD (paired doubles saja, {K_LAYERS} lapisan)')
print(f'   Strategi VQE : Optimasi ideal → Evaluasi noisy (2-fase)')


📌 Konfigurasi Benchmark CH₄ — k-UpCCSD + L-BFGS-B + Noise:
   Molekul      : CH₄  (C-H bond, simetri Td)
   Basis set    : STO-3G  →  18 qubit,  10 elektron
   Bond lengths : 0.80–2.00 Å  (17 titik)
   Optimizer    : L-BFGS-B  |  Restarts : 1  |  Seed : 42
   Backend noise: FakeSherbrooke (127-qubit, IBM)  |  Shots : 2,048
   Ansatz       : 1-UpCCSD (paired doubles saja, 1 lapisan)
   Strategi VQE : Optimasi ideal → Evaluasi noisy (2-fase)


In [4]:
# ════════════════════════════════════════════════════════════════════
# CELL 3 – ORCA LOADER  ← TIDAK DIUBAH DARI VERSI ASLI
# ════════════════════════════════════════════════════════════════════
import os, glob, re
import pandas as pd
from IPython.display import display

BASE_DIR        = os.path.join(os.path.expanduser('~'), 'Benchmarking VQE dan ORCA')
ORCA_OUTPUT_DIR = os.path.join(BASE_DIR, 'inputs_casscf')

cwd = os.getcwd()
print('=== DIAGNOSIS PATH ===')
print('Working directory :', cwd)
print('ORCA_OUTPUT_DIR   :', ORCA_OUTPUT_DIR)
print('Absolute path     :', os.path.abspath(ORCA_OUTPUT_DIR))
print('Folder exists?    :', os.path.isdir(os.path.abspath(ORCA_OUTPUT_DIR)))

CANDIDATE_DIRS = [
    ORCA_OUTPUT_DIR,
    BASE_DIR,
    os.path.join(cwd, 'Benchmarking VQE dan ORCA', 'inputs_casscf'),
    os.path.join(cwd, 'Benchmarking VQE dan ORCA'),
    os.path.join(cwd, 'inputs'),
    '.',
]

def is_orca_out(filepath):
    try:
        with open(filepath, 'r', errors='replace') as f:
            for i, line in enumerate(f):
                if 'O   R   C   A' in line or 'FINAL SINGLE POINT ENERGY' in line:
                    return True
                if i > 60:
                    break
    except Exception:
        pass
    return False


def find_orca_files(candidates):
    all_orca = []
    seen_dirs = set()
    for d in candidates:
        d = os.path.abspath(d)
        if d in seen_dirs or not os.path.isdir(d):
            continue
        seen_dirs.add(d)
        found = sorted(glob.glob(os.path.join(d, '*.out')))
        orca  = [f for f in found if is_orca_out(f)]
        if orca:
            print(f'  ✓ Ditemukan di: {d}  →  {len(orca)} file')
            all_orca.extend(orca)
        else:
            print(f'  ✗ Kosong      : {d}')
    return all_orca


def parse_bond_length_from_filename(filename):
    base = os.path.basename(filename)
    for m in re.findall(r'(\d+\.\d+)', base):
        val = float(m)
        if 0.10 <= val <= 9.99:
            return round(val, 4)
    for m in re.findall(r'[_\-](\d{2,4})(?=[^\d]|$)', base):
        val = float(m)
        while val >= 10:
            val /= 100
        if 0.10 <= val <= 9.99:
            return round(val, 4)
    return None


def parse_orca_energy(filepath):
    energy  = None
    pattern = re.compile(r'FINAL SINGLE POINT ENERGY\s+([\-\d.]+)')
    try:
        with open(filepath, 'r', errors='replace') as f:
            for line in f:
                m = pattern.search(line)
                if m:
                    energy = float(m.group(1))
    except Exception as e:
        print(f'  Gagal baca {os.path.basename(filepath)}: {e}')
    return energy


def load_orca_data(file_list):
    if not file_list:
        print('⚠ Tidak ada file ORCA .out untuk dibaca.')
        return {}
    data = {}
    rows = []
    for fp in sorted(file_list):
        fname = os.path.basename(fp)
        if fname.startswith('slurm') or fname.startswith('error'):
            continue
        R = parse_bond_length_from_filename(fp)
        E = parse_orca_energy(fp)
        if R is None:
            status = 'SKIP – R tidak terbaca'
        elif E is None:
            status = 'SKIP – energi tidak ada'
        else:
            status = 'OK'
            data[R] = E
        rows.append({'File': fname, 'R (Å)': R, 'E_ORCA (Ha)': E, 'Status': status})
    if rows:
        display(pd.DataFrame(rows))
    return dict(sorted(data.items()))


print('\nMencari file ORCA .out ...')
detected_files = find_orca_files(CANDIDATE_DIRS)

if not detected_files:
    print('\n⚠ Tidak ditemukan. Isi direktori yang relevan:')
    for d in [BASE_DIR, ORCA_OUTPUT_DIR, cwd]:
        if os.path.isdir(d):
            files = sorted(os.listdir(d))[:20]
            print(f'\n  [{d}]')
            for f in files:
                print(f'    {f}')
print('=====================')

orca_data = load_orca_data(detected_files)
print(f'\nTotal data ORCA terbaca: {len(orca_data)} titik')
if orca_data:
    print('Bond lengths (Å):', sorted(orca_data.keys()))

=== DIAGNOSIS PATH ===
Working directory : /mgpfs/home/mkhairiansyah
ORCA_OUTPUT_DIR   : /mgpfs/home/mkhairiansyah/Benchmarking VQE dan ORCA/inputs_casscf
Absolute path     : /mgpfs/home/mkhairiansyah/Benchmarking VQE dan ORCA/inputs_casscf
Folder exists?    : False

Mencari file ORCA .out ...
  ✗ Kosong      : /mgpfs/home/mkhairiansyah

⚠ Tidak ditemukan. Isi direktori yang relevan:

  [/mgpfs/home/mkhairiansyah]
    .bash_history
    .bash_logout
    .bash_profile
    .bashrc
    .bashrc.save
    .bashrc.save.1
    .cache
    .conda
    .condarc
    .config
    .copilot
    .dotnet
    .emacs
    .gitconfig
    .halo.py.swp
    .ipython
    .jupyter
    .keras
    .kshrc
    .lmod.d
⚠ Tidak ada file ORCA .out untuk dibaca.

Total data ORCA terbaca: 0 titik


In [5]:
# ════════════════════════════════════════════════════════════════════
# CELL 4 – HAMILTONIAN CH₄  (PySCF + OpenFermion → JW)
# ════════════════════════════════════════════════════════════════════
def get_ch4_geometry(bond_length):
    """
    Geometri tetrahedral CH₄ dengan C di origin.
    Semua 4 ikatan C-H = bond_length Å.

    Vektor arah H (satuan tetrahedral, |v|=1):
      H1 = (+1,+1,+1)/√3 , H2 = (+1,-1,-1)/√3
      H3 = (-1,+1,-1)/√3 , H4 = (-1,-1,+1)/√3
    """
    d = float(bond_length)
    s = d / np.sqrt(3.0)
    return (f'C  0 0 0; '
            f'H  {+s:.8f}  {+s:.8f}  {+s:.8f}; '
            f'H  {+s:.8f}  {-s:.8f}  {-s:.8f}; '
            f'H  {-s:.8f}  {+s:.8f}  {-s:.8f}; '
            f'H  {-s:.8f}  {-s:.8f}  {+s:.8f}')


def _build_h1(h1_mo, n_orb):
    """Bangun FermionOperator dari integrals satu-elektron (MO basis)."""
    H = FermionOperator()
    for p in range(n_orb):
        for q in range(n_orb):
            v = h1_mo[p, q]
            if abs(v) < 1e-12:
                continue
            for sigma in range(2):          # spin up (0) dan down (1)
                i, j = 2*p + sigma, 2*q + sigma
                H += FermionOperator(((i, 1), (j, 0)), v)
    return H


def _build_h2(eri_mo, n_orb):
    """Bangun FermionOperator dari integrals dua-elektron (MO basis)."""
    H = FermionOperator()
    for p in range(n_orb):
        for q in range(n_orb):
            for r in range(n_orb):
                for s in range(n_orb):
                    g = eri_mo[p, q, r, s]
                    if abs(g) < 1e-12:
                        continue
                    for sigma in range(2):
                        for tau in range(2):
                            H += FermionOperator(
                                ((2*p+sigma, 1), (2*r+tau,   1),
                                 (2*s+tau,   0), (2*q+sigma, 0)),
                                0.5 * g)
    return H


def build_hamiltonian_ch4(bond_length):
    """
    Bangun Hamiltonian CH₄ pada R_CH = bond_length Å.

    Returns
    -------
    H_mat      : ndarray        – matriks Hamiltonian elektronik (tanpa E_nuc)
    H_qubit_op : QubitOperator  – representasi qubit JW (tanpa E_nuc)
    E_nuc      : float          – energi repulsi inti (Ha)
    n_q        : int            – jumlah qubit = 18 untuk STO-3G/CH₄
    """
    mol         = gto.Mole()
    mol.atom    = get_ch4_geometry(bond_length)
    mol.basis   = 'sto-3g'
    mol.verbose = 0
    mol.build()

    mf    = scf.RHF(mol)
    mf.run()
    C     = mf.mo_coeff
    n_orb = C.shape[1]                  # 9 orbital untuk STO-3G/CH₄
    E_nuc = mol.energy_nuc()

    # Integrals dalam MO basis
    T_ao  = mol.intor('int1e_kin')
    V_ao  = mol.intor('int1e_nuc')
    h1_mo = C.T @ (T_ao + V_ao) @ C
    eri   = ao2mo.kernel(mol, C, compact=False).reshape(
                n_orb, n_orb, n_orb, n_orb)

    # FermionOperator → normal-ordering → Jordan-Wigner
    H_fermion  = normal_ordered(_build_h1(h1_mo, n_orb) + _build_h2(eri, n_orb))
    H_qubit_op = jordan_wigner(H_fermion)
    n_q        = count_qubits(H_qubit_op)       # harus 18

    # Matriks Hamiltonian (untuk FCI)
    H_mat_sparse = get_sparse_operator(H_qubit_op, n_qubits=n_q)

    return H_mat_sparse, H_qubit_op, E_nuc, n_q


# ── Uji cepat R = 1.09 Å ─────────────────────────────────────────
print('🔧 Uji Hamiltonian CH₄ pada R_CH = 1.09 Å (ekuilibrium) ...')
_H_sparse, _Hq, _Enuc, _nq = build_hamiltonian_ch4(1.09)
_evals, _evecs = eigsh(_H_sparse, k=1, which='SA')
print(f'   n_orb spasial = {_nq // 2}  (STO-3G, 5 occ + 4 virt)')
print(f'   n_qubits      = {_nq}')
print(f'   dim Hilbert   = 2^{_nq} = {2**_nq:,}')
print(f'   E_nuc         = {_Enuc:.6f} Ha')
print(f'   E_FCI         = {_evals[0] + _Enuc:.8f} Ha')
print('✅ Hamiltonian CH₄ OK')
del _H_sparse, _Hq, _Enuc, _nq, _evals, _evecs

🔧 Uji Hamiltonian CH₄ pada R_CH = 1.09 Å (ekuilibrium) ...


KeyboardInterrupt: 

In [6]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 – HELPER: OpenFermion QubitOperator → Qiskit SparsePauliOp
# ════════════════════════════════════════════════════════════════════
def qubitop_to_sparsepauliop(qubit_op, n_qubits):
    """
    Konversi OpenFermion QubitOperator → Qiskit SparsePauliOp.

    Pemetaan qubit yang konsisten antara circuit dan observable:
      OpenFermion (big-endian)  :  qubit k = bit ke-(n-1-k) dari indeks state
      Qiskit string             :  posisi k dalam string = Qiskit qubit (n-1-k)

    Dengan menempatkan Pauli OF qubit k di posisi string k:
      string[k] = Pauli pada OF qubit k = Pauli pada Qiskit qubit (n-1-k)
    → Konsisten dengan circuit di mana OF qubit k → Qiskit qubit (n-1-k).

    Term identitas () dari OF dipertahankan sebagai 'III...I'.
    """
    pauli_list = []
    for term, coeff in qubit_op.terms.items():
        if abs(coeff) < 1e-12:
            continue
        chars = ['I'] * n_qubits
        for of_q, p in term:
            chars[of_q] = p             # string[k] = Pauli pada OF qubit k
        pauli_list.append((''.join(chars), complex(coeff)))

    if not pauli_list:
        return SparsePauliOp.from_list([('I' * n_qubits, 0.0)])

    return SparsePauliOp.from_list(pauli_list, num_qubits=n_qubits).simplify()


print('✅ qubitop_to_sparsepauliop terdefinisi')

✅ qubitop_to_sparsepauliop terdefinisi


In [7]:
# ════════════════════════════════════════════════════════════════════
# CELL 6 – QISKIT k-UpCCSD CIRCUIT
#
# k-UpCCSD  (k-Unitary Pair Coupled Cluster Singles and Doubles)
# Perbedaan utama vs UCCSD:
#   • Doubles dibatasi ke "paired" (geminal): (i↑,i↓)→(a↑,a↓)
#     hanya eksitasi pasangan dalam orbital spasial yang sama
#   • k lapisan: ansatz diulang k kali dengan parameter independen
#   • Reduksi parameter: 1260 doubles UCCSD → 20 paired doubles (k=1)
# ════════════════════════════════════════════════════════════════════

def get_kupccsd_excitations(n_electrons, n_qubits):
    """
    Hasilkan daftar eksitasi untuk ansatz k-UpCCSD per lapisan.

    Singles        : semua kombinasi spin-orbital i (occ) → a (virt)
                     jumlah = n_occ × n_virt  (10 × 8 = 80 untuk CH₄)

    Paired doubles : hanya eksitasi pasangan geminal
                       (i↑, i↓) → (a↑, a↓)
                     yaitu spin-orbital (2p, 2p+1) → (2q, 2q+1)
                     untuk orbital spasial p (occ) dan q (virt).
                     jumlah = n_occ_spasial × n_virt_spasial
                            = 5 × 4 = 20 untuk CH₄ STO-3G

    Returns
    -------
    singles        : list[(i, a)]
    paired_doubles : list[(i, j, a, b)]   dengan j=i+1, b=a+1 (same-orbital pair)
    """
    n_occ         = n_electrons           # 10 spin-orbital terisi
    n_orb_spatial = n_qubits // 2         # 9  orbital spasial (STO-3G/CH₄)
    n_occ_spatial = n_electrons // 2      # 5  orbital spasial terisi

    # Singles : semua i → a (tidak ada batasan pairing)
    singles = [(i, a)
               for i in range(n_occ)
               for a in range(n_occ, n_qubits)]

    # Paired doubles : (i↑=2p, i↓=2p+1) → (a↑=2q, a↓=2q+1)
    paired_doubles = []
    for p in range(n_occ_spatial):
        for q in range(n_occ_spatial, n_orb_spatial):
            i, j = 2 * p,     2 * p + 1    # spin-up/down occupied
            a, b = 2 * q,     2 * q + 1    # spin-up/down virtual
            paired_doubles.append((i, j, a, b))

    return singles, paired_doubles


def _add_pauli_exp(qc, of_pauli_term, angle, n_qubits):
    """
    Tambahkan subcircuit exp(i · angle · P) ke QuantumCircuit.

    P  : Pauli string dari OpenFermion  (list of (of_qubit, pauli_char))
    angle : float atau Qiskit Parameter

    Pemetaan: OF qubit k → Qiskit qubit (n_qubits - 1 - k)
    (konsisten dengan qubitop_to_sparsepauliop dan HF state)

    Algoritma CNOT-ladder (implementasi standar):
      1. Ubah basis : X → H ,  Y → Sdg · H
      2. Rantai CNOT kiri→kanan  (akumulasi paritas)
      3. Rz(−2·angle) pada qubit terakhir  [ exp(iφZ) = Rz(−2φ) ]
      4. Inverse CNOT kanan→kiri
      5. Inverse basis change
    """
    # Map ke Qiskit qubit dan buang identitas
    active = sorted(
        [(n_qubits - 1 - of_q, p)
         for of_q, p in of_pauli_term if p != 'I'],
        key=lambda x: x[0]
    )
    if not active:
        return

    qs = [q for q, _ in active]
    ps = [p for _, p in active]

    # 1. Basis change
    for q, p in zip(qs, ps):
        if p == 'X':
            qc.h(q)
        elif p == 'Y':
            qc.sdg(q)
            qc.h(q)

    # 2. CNOT ladder
    for k in range(len(qs) - 1):
        qc.cx(qs[k], qs[k+1])

    # 3. Rz(-2·angle)  ←  exp(i·angle·Z)
    qc.rz(-2 * angle, qs[-1])

    # 4. Inverse CNOT
    for k in range(len(qs) - 2, -1, -1):
        qc.cx(qs[k], qs[k+1])

    # 5. Inverse basis change
    for q, p in zip(qs, ps):
        if p == 'X':
            qc.h(q)
        elif p == 'Y':
            qc.h(q)
            qc.s(q)


def build_kupccsd_circuit(n_qubits, n_electrons, k_layers=1):
    """
    Bangun Qiskit QuantumCircuit terparameterisasi untuk ansatz k-UpCCSD.

    State awal : |HF⟩  (dipersiapkan via gerbang X)
    Ansatz     : k lapisan berurutan, tiap lapisan terdiri dari:
                   (a) Single excitations   – semua i→a (80 untuk CH₄)
                   (b) Paired doubles       – hanya (i↑,i↓)→(a↑,a↓)
                                              yaitu 5×4=20 untuk CH₄
                 Tiap lapisan punya set parameter θ independen.

    Trotterisasi orde-1:
      exp(θ·G) ≈ Π_j exp(θ · i·c_j · P_j)
    di mana G = Σ_j (i·c_j) · P_j  (JW; c_j ∈ ℝ, P_j Hermitian Pauli)

    Parameters
    ----------
    n_qubits    : int – jumlah qubit    (18 untuk CH₄ STO-3G)
    n_electrons : int – jumlah elektron (10 untuk CH₄)
    k_layers    : int – jumlah lapisan  (default 1)

    Returns
    -------
    qc    : QuantumCircuit  – circuit terparameterisasi
    theta : ParameterVector – vektor parameter θ, panjang = k × (n_singles + n_paired)
    """
    singles, paired_doubles = get_kupccsd_excitations(n_electrons, n_qubits)
    n_params_per_layer = len(singles) + len(paired_doubles)
    n_params           = k_layers * n_params_per_layer
    theta              = ParameterVector('θ', n_params)
    qc                 = QuantumCircuit(n_qubits, name=f'k{k_layers}-UpCCSD_CH4')

    # ──────────────────────────────────────────────────────────────
    # 1. Persiapan state Hartree-Fock  |HF⟩
    #    OF qubit k (occ) → Qiskit qubit (n_qubits-1-k) → gerbang X
    # ──────────────────────────────────────────────────────────────
    for i in range(n_electrons):
        qc.x(n_qubits - 1 - i)
    qc.barrier(label='|HF⟩')

    # ──────────────────────────────────────────────────────────────
    # 2. k lapisan k-UpCCSD  (parameter independen per lapisan)
    # ──────────────────────────────────────────────────────────────
    for layer in range(k_layers):
        param_offset = layer * n_params_per_layer
        param_idx    = param_offset

        # ── 2a. Single excitations ──────────────────────────────
        #    G_ia = a†_a a_i  −  a†_i a_a  (anti-Hermitian, JW imag coeff)
        for i, a in singles:
            gen_f = (FermionOperator(((a, 1), (i, 0))) -
                     FermionOperator(((i, 1), (a, 0))))
            gen_q = jordan_wigner(normal_ordered(gen_f))
            for pauli_term, coeff in gen_q.terms.items():
                if abs(coeff) < 1e-12 or not pauli_term:
                    continue
                c_im = float(np.imag(coeff))
                if abs(c_im) > 1e-12:
                    _add_pauli_exp(qc, list(pauli_term),
                                   c_im * theta[param_idx], n_qubits)
            param_idx += 1
        qc.barrier(label=f'singles L{layer + 1}')

        # ── 2b. Paired double excitations ───────────────────────
        #    G_ijab = a†_a a†_b a_j a_i  −  h.c.
        #    Batasan: (i,j) = (2p, 2p+1)  dan  (a,b) = (2q, 2q+1)
        #    yaitu kedua elektron dari / ke orbital spasial yang sama
        for i, j, a, b in paired_doubles:
            gen_f = (FermionOperator(((a, 1), (b, 1), (j, 0), (i, 0))) -
                     FermionOperator(((i, 1), (j, 1), (b, 0), (a, 0))))
            gen_q = jordan_wigner(normal_ordered(gen_f))
            for pauli_term, coeff in gen_q.terms.items():
                if abs(coeff) < 1e-12 or not pauli_term:
                    continue
                c_im = float(np.imag(coeff))
                if abs(c_im) > 1e-12:
                    _add_pauli_exp(qc, list(pauli_term),
                                   c_im * theta[param_idx], n_qubits)
            param_idx += 1

        if layer < k_layers - 1:
            qc.barrier(label=f'paired doubles L{layer + 1}')

    qc.barrier(label='paired doubles')
    return qc, theta


# ── Build circuit SEKALI (dipakai ulang untuk semua bond lengths) ─────────
_singles, _paired_doubles = get_kupccsd_excitations(N_ELECTRONS, N_QUBITS)
N_PARAMS_PER_LAYER = len(_singles) + len(_paired_doubles)
N_PARAMS           = K_LAYERS * N_PARAMS_PER_LAYER

print(f'✅ Konfigurasi {K_LAYERS}-UpCCSD (CH₄, STO-3G):')
print(f'   Occupied spin-orbital      : {N_ELECTRONS}  (5 spasial × 2 spin)')
print(f'   Virtual  spin-orbital      : {N_QUBITS - N_ELECTRONS}  (4 spasial × 2 spin)')
print(f'   Single excitations         : {len(_singles)}')
print(f'   Paired double excitations  : {len(_paired_doubles)}  (vs {len(_singles)*(len(_singles)-1)//2 - sum(1 for x in range(10) for y in range(x+1,10) if x < 10 and y < 10)} UCCSD full doubles)')
print(f'   Parameter per lapisan      : {N_PARAMS_PER_LAYER}')
print(f'   Lapisan (k)                : {K_LAYERS}')
print(f'   Total parameter θ          : {N_PARAMS}  (vs 1340 UCCSD)')
print()
print(f'⏳ Membangun Qiskit QuantumCircuit {K_LAYERS}-UpCCSD ({N_PARAMS} param, {K_LAYERS} lapisan) ...')
print('   (k-UpCCSD jauh lebih cepat dibangun vs UCCSD)')

_t0 = time.time()
KUPCCSD_CIRCUIT, THETA_PARAMS = build_kupccsd_circuit(N_QUBITS, N_ELECTRONS, K_LAYERS)
_dt = time.time() - _t0

print(f'✅ Circuit berhasil dibangun dalam {_dt:.1f} detik')
print(f'   Jumlah qubit               : {KUPCCSD_CIRCUIT.num_qubits}')
print(f'   Jumlah parameter           : {KUPCCSD_CIRCUIT.num_parameters}')
print(f'   Depth circuit (pre-transpile): {KUPCCSD_CIRCUIT.depth()}')
print(f'   Gate count                 : {dict(KUPCCSD_CIRCUIT.count_ops())}')


✅ Konfigurasi 1-UpCCSD (CH₄, STO-3G):
   Occupied spin-orbital      : 10  (5 spasial × 2 spin)
   Virtual  spin-orbital      : 8  (4 spasial × 2 spin)
   Single excitations         : 80
   Paired double excitations  : 20  (vs 3115 UCCSD full doubles)
   Parameter per lapisan      : 100
   Lapisan (k)                : 1
   Total parameter θ          : 100  (vs 1340 UCCSD)

⏳ Membangun Qiskit QuantumCircuit 1-UpCCSD (100 param, 1 lapisan) ...
   (k-UpCCSD jauh lebih cepat dibangun vs UCCSD)
✅ Circuit berhasil dibangun dalam 0.1 detik
   Jumlah qubit               : 18
   Jumlah parameter           : 100
   Depth circuit (pre-transpile): 4038
   Gate count                 : {'cx': 3840, 'h': 1920, 'sdg': 480, 's': 480, 'rz': 320, 'x': 10, 'barrier': 3}


In [8]:
# ════════════════════════════════════════════════════════════════════
# CELL 7 – NOISE MODEL (FAKEMUMBAI) + SETUP ESTIMATOR
# ════════════════════════════════════════════════════════════════════

# ── Noise model dari FakeMumbai ───────────────────────────────────
print(f'⏳ Memuat noise model dari {BACKEND_NAME} ...')
fake_backend = _FakeBackendCls()
noise_model  = NoiseModel.from_backend(fake_backend)

print(f'✅ Noise Model dari {BACKEND_NAME}:')
print(f'   Basis gates  : {noise_model.basis_gates}')
print(f'   Qubit noise  : {len(noise_model.noise_qubits)} qubit terpengaruh')
print(f'   (CH₄ memakai 18 dari 27 qubit FakeMumbai)')
print()

# ──────────────────────────────────────────────────────────────────
# Estimator IDEAL
# AerEstimator tanpa noise_model → simulasi statevector eksak
# Digunakan untuk: optimasi parameter θ*
# ──────────────────────────────────────────────────────────────────
estimator_ideal = AerEstimator(
    backend_options = {"method": "statevector"},
)
print('✅ Estimator ideal  : AerEstimator (statevector, tanpa noise)')
print('   Digunakan untuk  : optimasi parameter θ*')

# ──────────────────────────────────────────────────────────────────
# Estimator NOISY (FakeMumbai + 16 384 shots)
# Digunakan untuk: evaluasi ⟨H⟩ pada θ* dengan noise hardware IBM
# Transpilasi ke topologi FakeMumbai dengan optimization_level=1
# ──────────────────────────────────────────────────────────────────
estimator_noisy = AerEstimator(
    backend_options   = {
        "noise_model" : noise_model,
        "basis_gates" : noise_model.basis_gates,
    },
    run_options       = {"shots": N_SHOTS},
    transpile_options = {"optimization_level": 1},
)
print(f'✅ Estimator noisy  : AerEstimator (FakeMumbai, {N_SHOTS:,} shots)')
print('   Digunakan untuk  : evaluasi ⟨H(θ*)⟩ dengan noise hardware')

⏳ Memuat noise model dari FakeSherbrooke (127-qubit, IBM) ...


✅ Noise Model dari FakeSherbrooke (127-qubit, IBM):
   Basis gates  : ['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']
   Qubit noise  : 27 qubit terpengaruh
   (CH₄ memakai 18 dari 27 qubit FakeMumbai)

✅ Estimator ideal  : AerEstimator (statevector, tanpa noise)
   Digunakan untuk  : optimasi parameter θ*
✅ Estimator noisy  : AerEstimator (FakeMumbai, 2,048 shots)
   Digunakan untuk  : evaluasi ⟨H(θ*)⟩ dengan noise hardware


In [9]:
# ════════════════════════════════════════════════════════════════════
# CELL 8 – VQE RUNNER  (Qiskit circuit-based, dua-fase)
#           Optimizer: L-BFGS-B  (gradient via 2-point finite differences)
# ════════════════════════════════════════════════════════════════════

def run_vqe_qiskit(circuit, H_pauli, n_params,
                   est_ideal, est_noisy,
                   method='L-BFGS-B', n_restarts=1, seed=42):
    """
    Jalankan VQE berbasis Qiskit QuantumCircuit dengan strategi dua-fase.

    ┌──────────────────────────────────────────────────────────────┐
    │  Fase 1 – Optimasi Ideal                                     │
    │  Gunakan est_ideal (statevector, tanpa noise) untuk          │
    │  mencari θ* yang meminimalkan ⟨ψ(θ)|H|ψ(θ)⟩.                 │
    │  Optimizer: L-BFGS-B (quasi-Newton, gradient via 2-point FD) │
    │  Keunggulan: konvergensi cepat, lebih sedikit iterasi.       │
    │                                                              │
    │  Fase 2 – Evaluasi Noisy                                     │
    │  Hitung ⟨H⟩ pada θ* menggunakan est_noisy (FakeSherbrooke +  │
    │  16 384 shots). Rata-ratakan 3× untuk meredam shot noise.    │
    │  Selisih E_noisy − E_ideal ≈ dampak noise hardware IBM.      │
    └──────────────────────────────────────────────────────────────┘

    Catatan L-BFGS-B:
      • Gradient dihitung via finite differences 2-point (scipy)
      • step  eps = 1e-4  dipilih sesuai skala energi kimia (Ha)
      • Setiap langkah gradient ≈ 2 × n_params evaluasi fungsi biaya
      • k-UpCCSD (n_params=100) jauh lebih efisien vs UCCSD (1340)

    Parameters
    ----------
    circuit    : QuantumCircuit terparameterisasi (k-UpCCSD)
    H_pauli    : SparsePauliOp  (Hamiltonian elektronik)
    n_params   : int
    est_ideal  : AerEstimator (tanpa noise)
    est_noisy  : AerEstimator (FakeSherbrooke noise)
    method     : str  ('L-BFGS-B' / 'COBYLA' / 'BFGS')
    n_restarts : int
    seed       : int

    Returns
    -------
    dict:
      E_ideal        – energi VQE ideal (elektronik, Ha)
      E_noisy        – energi VQE noisy (elektronik, Ha)
      optimal_params – θ* hasil optimasi
      n_iters        – total evaluasi fungsi biaya
    """
    rng = np.random.RandomState(seed)

    # ── Opsi optimizer ────────────────────────────────────────────────
    # L-BFGS-B: ftol=tol konvergensi nilai fungsi, gtol=tol norma grad,
    #           eps=step finite-diff (1e-4 cocok untuk skala energi Ha),
    #           maxcor=jumlah vektor memori quasi-Newton
    OPTS = {
        'L-BFGS-B': {'maxiter': 2000, 'ftol': 1e-10, 'gtol': 1e-7,
                     'eps': 1e-4,  'maxcor': 20},
        'COBYLA'  : {'maxiter': 5000, 'rhobeg': 0.1},
        'BFGS'    : {'maxiter': 2000, 'gtol'  : 1e-6},
    }
    opts = OPTS.get(method, {'maxiter': 5000})

    # Gunakan 2-point finite differences untuk gradient bila L-BFGS-B
    # (scipy menghitung otomatis; COBYLA tidak memerlukan gradient)
    jac_method = '2-point' if method in ('L-BFGS-B', 'BFGS') else None

    best = {'E_ideal': np.inf, 'params': None, 'n_iters': 0}

    # ────────────────────────────────────────────────────────────
    # Fase 1 : Optimasi dengan estimator ideal
    # ────────────────────────────────────────────────────────────
    for trial in range(n_restarts):
        history = []

        def cost_fn(p):
            try:
                job = est_ideal.run([circuit], [H_pauli], [p.tolist()])
                E   = float(job.result().values[0])
            except Exception as err:
                print(f'    ⚠ Ideal eval gagal: {err}')
                E   = 0.0
            history.append(E)
            return E

        # Inisialisasi lebih dekat ke HF (θ=0) untuk L-BFGS-B
        theta0 = rng.uniform(-0.05, 0.05, n_params)
        res    = minimize(cost_fn, theta0,
                          method=method,
                          jac=jac_method,
                          options=opts)

        if res.fun < best['E_ideal']:
            best.update({
                'E_ideal': res.fun,
                'params' : res.x.copy(),
                'n_iters': len(history)
            })

    # ────────────────────────────────────────────────────────────
    # Fase 2 : Evaluasi noisy pada θ*
    # ────────────────────────────────────────────────────────────
    E_noisy = np.nan
    if best['params'] is not None:
        try:
            vals = []
            for _ in range(3):    # 3× averaging untuk meredam shot noise
                job = est_noisy.run([circuit], [H_pauli],
                                    [best['params'].tolist()])
                vals.append(float(job.result().values[0]))
            E_noisy = float(np.mean(vals))
        except Exception as err:
            print(f'    ⚠ Noisy eval gagal: {err}')

    return {
        'E_ideal'       : best['E_ideal'],
        'E_noisy'       : E_noisy,
        'optimal_params': best['params'],
        'n_iters'       : best['n_iters'],
    }


print('✅ run_vqe_qiskit terdefinisi (L-BFGS-B + 2-point FD gradient)')
print(f'   Optimizer   : {METHOD_VQE}')
print(f'   Gradient    : 2-point finite differences  (eps=1e-4)')
print(f'   ∂E cost/step: ~2 × {N_PARAMS} = {2*N_PARAMS} eval fungsi per iterasi gradient')


✅ run_vqe_qiskit terdefinisi (L-BFGS-B + 2-point FD gradient)
   Optimizer   : L-BFGS-B
   Gradient    : 2-point finite differences  (eps=1e-4)
   ∂E cost/step: ~2 × 100 = 200 eval fungsi per iterasi gradient


In [10]:
print(KUPCCSD_CIRCUIT.depth())
print(KUPCCSD_CIRCUIT.count_ops())

4038
OrderedDict([('cx', 3840), ('h', 1920), ('sdg', 480), ('s', 480), ('rz', 320), ('x', 10), ('barrier', 3)])


In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 9 – LOOP VQE: SEMUA PANJANG IKATAN C-H
# ════════════════════════════════════════════════════════════════════
results_vqe = []
n_total     = len(BOND_LENGTHS_VQE)
SEP         = '═' * 72

print(SEP)
print(f'  Benchmark VQE k-UpCCSD — CH₄ STO-3G  ({n_total} titik ikatan C-H)')
print(f'  Optimizer : {METHOD_VQE}  |  k-layers : {K_LAYERS}  |  Restarts : {N_RESTARTS}  |  Shots : {N_SHOTS:,}')
print(f'  Backend   : {BACKEND_NAME}')
print(SEP)

t_total_start = time.time()

for idx_R, R in enumerate(BOND_LENGTHS_VQE):
    t_R_start = time.time()
    print(f'\n  [{idx_R+1:2d}/{n_total}] R_CH = {R:.2f} Å', flush=True)

    try:
        # ── 1. Hamiltonian CH₄ ─────────────────────────────────────
        print('       Membangun Hamiltonian ...', end='', flush=True)
        H_mat, H_qubit_op, E_nuc, n_q = build_hamiltonian_ch4(R)
        H_pauli = qubitop_to_sparsepauliop(H_qubit_op, n_q)
        print(f' OK  ({len(H_pauli)} suku Pauli)')

        # ── 2. FCI (diagonalisasi matriks eksak, NumPy) ─────────────
        evals, evecs = eigsh(H_mat, k=1, which='SA')
        E_fci = evals[0] + E_nuc

        # ── 3. HF (θ = 0, titik awal ansatz) ────────────────────────
        theta_zero = np.zeros(N_PARAMS)
        job_hf     = estimator_ideal.run(
            [KUPCCSD_CIRCUIT], [H_pauli], [theta_zero.tolist()])
        E_hf       = float(job_hf.result().values[0]) + E_nuc

        # ── 4. VQE  (dua-fase: ideal optimasi → noisy evaluasi) ──────
        print('       Menjalankan VQE ...', end='', flush=True)
        vqe = run_vqe_qiskit(
            KUPCCSD_CIRCUIT, H_pauli, N_PARAMS,
            estimator_ideal, estimator_noisy,
            method=METHOD_VQE, n_restarts=N_RESTARTS, seed=SEED
        )
        print(' OK')

        E_vqe_ideal = vqe['E_ideal'] + E_nuc
        E_vqe_noisy = (vqe['E_noisy'] + E_nuc
                       if not np.isnan(vqe['E_noisy']) else np.nan)

        dE_ideal    = abs(E_vqe_ideal - E_fci)
        dE_noisy    = (abs(E_vqe_noisy - E_fci)
                       if not np.isnan(E_vqe_noisy) else np.nan)
        ca          = '✓' if dE_ideal < 1.6e-3 else '✗'
        noise_impact = abs(E_vqe_noisy - E_vqe_ideal) if not np.isnan(E_vqe_noisy) else np.nan

        results_vqe.append({
            'R'           : R,
            'E_fci'       : E_fci,
            'E_hf'        : E_hf,
            'E_vqe_ideal' : E_vqe_ideal,
            'E_vqe_noisy' : E_vqe_noisy,
            'dE_ideal'    : dE_ideal,
            'dE_noisy'    : dE_noisy,
            'noise_impact': noise_impact,
            'n_iters'     : vqe['n_iters'],
        })

        dt = time.time() - t_R_start
        print(f'       E_FCI       = {E_fci:+.6f} Ha')
        print(f'       E_HF (θ=0)  = {E_hf:+.6f} Ha')
        print(f'       E_VQE ideal = {E_vqe_ideal:+.6f} Ha   |ΔE| = {dE_ideal:.2e}  [{ca}]')
        e_noisy_str = f'{E_vqe_noisy:+.6f}' if not np.isnan(E_vqe_noisy) else 'N/A'
        dE_noisy_str= f'{dE_noisy:.2e}'     if not np.isnan(dE_noisy)    else 'N/A'
        ni_str      = f'{noise_impact:.2e}' if not np.isnan(noise_impact) else 'N/A'
        print(f'       E_VQE noisy = {e_noisy_str} Ha   |ΔE| = {dE_noisy_str}  [noise: {ni_str}]')
        print(f'       Iter VQE    = {vqe["n_iters"]:,}   |   Waktu: {dt:.1f} s')

    except Exception as exc:
        print(f'       ❌ GAGAL : {exc}')

t_total = time.time() - t_total_start
print(f'\n{SEP}')
print(f'  Selesai — {len(results_vqe)}/{n_total} titik berhasil')
print(f'  Total waktu : {t_total/60:.1f} menit')
print(SEP)

════════════════════════════════════════════════════════════════════════
  Benchmark VQE k-UpCCSD — CH₄ STO-3G  (17 titik ikatan C-H)
  Optimizer : L-BFGS-B  |  k-layers : 1  |  Restarts : 1  |  Shots : 2,048
  Backend   : FakeSherbrooke (127-qubit, IBM)
════════════════════════════════════════════════════════════════════════

  [ 1/17] R_CH = 0.80 Å
       Membangun Hamiltonian ...

 OK  (6892 suku Pauli)
       Menjalankan VQE ...


KeyboardInterrupt



In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 10 – TABEL HASIL BENCHMARK
# ════════════════════════════════════════════════════════════════════
rows = []
for r in results_vqe:
    ca           = '✓' if r['dE_ideal'] < 1.6e-3 else '✗'
    noisy_E_str  = (f"{r['E_vqe_noisy']:.8f}"
                    if not np.isnan(r['E_vqe_noisy']) else '—')
    dE_noisy_str = (f"{r['dE_noisy']:.3e}"
                    if not np.isnan(r['dE_noisy'])    else '—')
    ni_str       = (f"{r['noise_impact']:.3e}"
                    if not np.isnan(r['noise_impact']) else '—')

    rows.append({
        'R (Å)'              : round(r['R'],           2),
        'E_FCI (Ha)'         : round(r['E_fci'],       8),
        'E_HF (Ha)'          : round(r['E_hf'],        8),
        'E_VQE ideal (Ha)'   : round(r['E_vqe_ideal'], 8),
        '|ΔE_ideal| (Ha)'    : f"{r['dE_ideal']:.3e}",
        'ChAcc'              : ca,
        'E_VQE noisy (Ha)'   : noisy_E_str,
        '|ΔE_noisy| (Ha)'    : dE_noisy_str,
        'Noise Impact (Ha)'  : ni_str,
        'Iter VQE'           : r['n_iters'],
    })

df_result = pd.DataFrame(rows)

def _color_ca(v):
    return 'color: green; font-weight: bold' if v == '✓' else 'color: red'

styled = (df_result.style
    .applymap(_color_ca, subset=['ChAcc'])
    .set_caption(
        '📊 Benchmark VQE k-UpCCSD — CH₄ STO-3G  |  '
        'L-BFGS-B | Ideal (Statevector) vs Noisy (FakeSherbrooke 16 384 shots) vs FCI')
    .set_properties(**{'text-align': 'center'})
    .hide(axis='index'))
display(styled)

# ── Statistik ─────────────────────────────────────────────────────
dE_ideal_all  = [r['dE_ideal']     for r in results_vqe]
dE_noisy_all  = [r['dE_noisy']     for r in results_vqe
                 if not np.isnan(r['dE_noisy'])]
ni_all        = [r['noise_impact'] for r in results_vqe
                 if not np.isnan(r['noise_impact'])]
n_ok          = sum(1 for d in dE_ideal_all if d < 1.6e-3)

print(f'\n📈 Statistik |ΔE_VQE ideal| vs FCI:')
print(f'   Rata-rata : {np.mean(dE_ideal_all):.3e} Ha')
print(f'   Maks      : {np.max(dE_ideal_all):.3e} Ha  (R={results_vqe[np.argmax(dE_ideal_all)]["R"]:.2f} Å)')
print(f'   Min       : {np.min(dE_ideal_all):.3e} Ha  (R={results_vqe[np.argmin(dE_ideal_all)]["R"]:.2f} Å)')
print(f'   Chemical accuracy: {n_ok}/{len(dE_ideal_all)} titik ({100*n_ok/len(dE_ideal_all):.0f}%)')
if dE_noisy_all:
    print(f'\n📈 Statistik |ΔE_VQE noisy| vs FCI:')
    print(f'   Rata-rata : {np.mean(dE_noisy_all):.3e} Ha')
    print(f'   Maks      : {np.max(dE_noisy_all):.3e} Ha')
if ni_all:
    print(f'\n📈 Dampak Noise Hardware  |E_noisy − E_ideal|:')
    print(f'   Rata-rata : {np.mean(ni_all):.3e} Ha')
    print(f'   Maks      : {np.max(ni_all):.3e} Ha')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 11 – VISUALISASI  (VQE only: FCI + Ideal + Noisy)
#            Plot ORCA ditiadakan karena data belum tersedia
# ════════════════════════════════════════════════════════════════════
R_arr         = np.array([r['R']            for r in results_vqe])
E_fci_arr     = np.array([r['E_fci']        for r in results_vqe])
E_hf_arr      = np.array([r['E_hf']         for r in results_vqe])
E_ideal_arr   = np.array([r['E_vqe_ideal']  for r in results_vqe])
E_noisy_arr   = np.array([r['E_vqe_noisy']  for r in results_vqe])
dE_i_arr      = np.array([r['dE_ideal']     for r in results_vqe])
dE_n_arr      = np.array([r['dE_noisy']     for r in results_vqe])
ni_arr        = np.array([r['noise_impact'] for r in results_vqe])
iters_arr     = np.array([r['n_iters']      for r in results_vqe])
mask_noisy    = ~np.isnan(E_noisy_arr)

# ── Palet warna ───────────────────────────────────────────────────
C = {
    'fci'    : '#1a1a2e',
    'hf'     : '#e07c00',
    'ideal'  : '#0077b6',
    'noisy'  : '#c1121f',
    'chem'   : '#52b788',
    'ni'     : '#9b5de5',
    'grid'   : '#dee2e6',
}

fig = plt.figure(figsize=(18, 15))
fig.patch.set_facecolor('#f8f9fa')
gs  = gridspec.GridSpec(3, 3, hspace=0.50, wspace=0.38,
                        top=0.91, bottom=0.06, left=0.07, right=0.97)

# ──────────────────────────────────────────────────────────────────
# Plot 1: PES Utama  (FCI + VQE Ideal + VQE Noisy)
# ──────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor('#ffffff')
ax1.plot(R_arr, E_fci_arr, '-',
         color=C['fci'],   lw=2.5, zorder=5,
         label='FCI Eksak (STO-3G, diagonalisasi matriks)')
ax1.plot(R_arr, E_ideal_arr, 'o--',
         color=C['ideal'], lw=2.2, ms=7, zorder=4,
         label='VQE k-UpCCSD Ideal (Qiskit circuit, statevector)')
if mask_noisy.any():
    ax1.plot(R_arr[mask_noisy], E_noisy_arr[mask_noisy], 's-.',
             color=C['noisy'], lw=2.0, ms=7, zorder=3,
             label=f'VQE k-UpCCSD Noisy (FakeSherbrooke IBM, {N_SHOTS:,} shots)')

# Tandai minimum FCI
idx_min = np.argmin(E_fci_arr)
ax1.axvline(R_arr[idx_min], color='gray', ls='--', alpha=0.4, lw=1.5)
ax1.scatter([R_arr[idx_min]], [E_fci_arr[idx_min]],
            s=200, color=C['fci'], zorder=10,
            edgecolor='white', lw=2.5)
ax1.annotate(
    f'R$_{{\\mathrm{{eq}}}}$ = {R_arr[idx_min]:.2f} Å\nE$_{{\\mathrm{{FCI}}}}$ = {E_fci_arr[idx_min]:.5f} Ha',
    xy=(R_arr[idx_min], E_fci_arr[idx_min]),
    xytext=(R_arr[idx_min]+0.18, E_fci_arr[idx_min]-0.12),
    fontsize=10.5,
    bbox=dict(boxstyle='round,pad=0.4', fc='#fff3cd', alpha=0.9, ec='#c9a94b'),
    arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

ax1.set_xlabel('Panjang Ikatan C-H (Å)', fontsize=13)
ax1.set_ylabel('Energi Total (Hartree)', fontsize=13)
ax1.set_title(
    f'Benchmark Kurva Energi Potensial CH₄ — '
    f'VQE {K_LAYERS}-UpCCSD Ideal vs Noisy (FakeSherbrooke IBM) vs FCI',
    fontsize=13, weight='bold', pad=10)
ax1.legend(fontsize=11, loc='upper right', framealpha=0.95)
ax1.grid(True, color=C['grid'], alpha=0.7)
ax1.set_xlim(R_arr[0]-0.02, R_arr[-1]+0.02)

# ──────────────────────────────────────────────────────────────────
# Plot 2: |ΔE_ideal| vs FCI  (semilogy)
# ──────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor('#ffffff')
ax2.semilogy(R_arr, dE_i_arr, 'o-',
             color=C['ideal'], lw=2.2, ms=7,
             label='|E_VQE ideal − E_FCI|')
ax2.axhline(1.6e-3, color=C['chem'], ls='--', lw=2, alpha=0.9,
            label='Chemical accuracy (1.6 mHa)')
ax2.fill_between(R_arr, 0, 1.6e-3, alpha=0.08, color=C['chem'])
ax2.set_xlabel('Panjang Ikatan C-H (Å)', fontsize=12)
ax2.set_ylabel('|ΔE| (Ha)', fontsize=12)
ax2.set_title('Error VQE Ideal vs FCI', fontsize=12, weight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.4, which='both', color=C['grid'])

# ──────────────────────────────────────────────────────────────────
# Plot 3: |ΔE_noisy| vs FCI  (semilogy)
# ──────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor('#ffffff')
if mask_noisy.any():
    ax3.semilogy(R_arr[mask_noisy], dE_n_arr[mask_noisy], 's-',
                 color=C['noisy'], lw=2.2, ms=7,
                 label='|E_VQE noisy − E_FCI|')
    ax3.axhline(1.6e-3, color=C['chem'], ls='--', lw=2, alpha=0.9,
                label='Chemical accuracy')
    ax3.fill_between(R_arr[mask_noisy], 0, 1.6e-3,
                     alpha=0.08, color=C['chem'])
    ax3.legend(fontsize=9)
else:
    ax3.text(0.5, 0.5, 'Data noisy\nbelum tersedia',
             ha='center', va='center', transform=ax3.transAxes,
             fontsize=13, color='gray')
ax3.set_xlabel('Panjang Ikatan C-H (Å)', fontsize=12)
ax3.set_ylabel('|ΔE| (Ha)', fontsize=12)
ax3.set_title('Error VQE Noisy vs FCI', fontsize=12, weight='bold')
ax3.grid(True, alpha=0.4, which='both', color=C['grid'])

# ──────────────────────────────────────────────────────────────────
# Plot 4: Noise Impact  |E_noisy − E_ideal|
# ──────────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor('#ffffff')
if mask_noisy.any():
    ax4.semilogy(R_arr[mask_noisy], ni_arr[mask_noisy], '^-',
                 color=C['ni'], lw=2.2, ms=7,
                 label='|E_noisy − E_ideal|')
    ax4.axhline(1.6e-3, color=C['chem'], ls='--', lw=2, alpha=0.9,
                label='Chemical accuracy')
    avg_ni = ni_arr[mask_noisy].mean()
    ax4.axhline(avg_ni, color=C['ni'], ls=':', lw=1.5, alpha=0.7,
                label=f'Rata-rata: {avg_ni:.2e} Ha')
    ax4.fill_between(R_arr[mask_noisy], 0, 1.6e-3,
                     alpha=0.08, color=C['chem'])
    ax4.legend(fontsize=9)
else:
    ax4.text(0.5, 0.5, 'Data noisy\nbelum tersedia',
             ha='center', va='center', transform=ax4.transAxes,
             fontsize=13, color='gray')
ax4.set_xlabel('Panjang Ikatan C-H (Å)', fontsize=12)
ax4.set_ylabel('|ΔE| (Ha)', fontsize=12)
ax4.set_title('Dampak Noise Hardware\n|E_VQE noisy − E_VQE ideal|',
              fontsize=12, weight='bold')
ax4.grid(True, alpha=0.4, which='both', color=C['grid'])

# ──────────────────────────────────────────────────────────────────
# Plot 5: Iterasi VQE per titik
# ──────────────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
ax5.set_facecolor('#ffffff')
ax5.bar(R_arr, iters_arr, width=0.04,
        color=C['ideal'], alpha=0.7, edgecolor='navy')
avg_iter = iters_arr.mean()
ax5.axhline(avg_iter, color='red', ls='--', lw=1.5, alpha=0.8,
            label=f'Rata-rata: {avg_iter:.0f}')
ax5.set_xlabel('Panjang Ikatan C-H (Å)', fontsize=12)
ax5.set_ylabel('Iterasi Optimizer', fontsize=12)
ax5.set_title('Iterasi VQE per Titik Ikatan', fontsize=12, weight='bold')
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.4, axis='y', color=C['grid'])

# ──────────────────────────────────────────────────────────────────
# Plot 6: Perbandingan |ΔE| ideal vs noisy  (overlay)
# ──────────────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
ax6.set_facecolor('#ffffff')
ax6.semilogy(R_arr, dE_i_arr, 'o-',
             color=C['ideal'], lw=2.2, ms=6, alpha=0.9,
             label='Ideal vs FCI')
if mask_noisy.any():
    ax6.semilogy(R_arr[mask_noisy], dE_n_arr[mask_noisy], 's--',
                 color=C['noisy'], lw=2.2, ms=6, alpha=0.9,
                 label='Noisy vs FCI')
ax6.axhline(1.6e-3, color=C['chem'], ls='--', lw=2, alpha=0.9,
            label='Chemical accuracy')
ax6.fill_between(R_arr, 0, 1.6e-3, alpha=0.08, color=C['chem'])
ax6.set_xlabel('Panjang Ikatan C-H (Å)', fontsize=12)
ax6.set_ylabel('|ΔE| (Ha)', fontsize=12)
ax6.set_title('Perbandingan Error:\nIdeal vs Noisy (overlay)', fontsize=12, weight='bold')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.4, which='both', color=C['grid'])

# ──────────────────────────────────────────────────────────────────
# Plot 7: Teks info ringkasan
# ──────────────────────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
ax7.set_facecolor('#edf2fb')
ax7.axis('off')

_n_ok = sum(1 for d in dE_i_arr if d < 1.6e-3)
_pct  = 100 * _n_ok / len(dE_i_arr)
_mean_i = np.mean(dE_i_arr)
_mean_n = np.mean(dE_n_arr[mask_noisy]) if mask_noisy.any() else float('nan')
_mean_ni= np.mean(ni_arr[mask_noisy])   if mask_noisy.any() else float('nan')

summary = (
    f"📋  Ringkasan Benchmark\n"
    f"════════════════════════════\n"
    f"Molekul  : CH₄  (STO-3G)\n"
    f"Qubit    : {N_QUBITS}   Elektron: {N_ELECTRONS}\n"
    f"Param θ  : {N_PARAMS:,}\n"
    f"Backend  : FakeMumbai (IBM)\n"
    f"Shots    : {N_SHOTS:,}\n"
    f"\n"
    f"── VQE Ideal ──\n"
    f"⟨|ΔE|⟩  = {_mean_i:.3e} Ha\n"
    f"ChAcc   = {_n_ok}/{len(dE_i_arr)}  ({_pct:.0f}%)\n"
    f"\n"
    f"── VQE Noisy ──\n"
    f"⟨|ΔE|⟩  = {_mean_n:.3e} Ha\n"
    f"\n"
    f"── Noise Impact ──\n"
    f"⟨|ΔE_noise|⟩ = {_mean_ni:.3e} Ha"
)
ax7.text(0.05, 0.95, summary,
         transform=ax7.transAxes,
         fontsize=10.5, fontfamily='monospace',
         verticalalignment='top',
         bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# ── Super title ───────────────────────────────────────────────────
plt.suptitle(
    f'Benchmark VQE {K_LAYERS}-UpCCSD — CH₄ STO-3G (18 qubit, {N_PARAMS} param, k={K_LAYERS})\n'
    f'L-BFGS-B | FakeSherbrooke Noise Model | '
    f'{N_SHOTS:,} shots  —  ORCA: belum diproses',
    fontsize=12.5, weight='bold', y=0.995)

plt.savefig(f'benchmark_vqe_CH4_k{K_LAYERS}UpCCSD_LBFGSB.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'✅ Plot disimpan: benchmark_vqe_CH4_k{K_LAYERS}UpCCSD_LBFGSB.png')